# 09 — Final model comparison and ranking

This notebook compares one final representative from each modelling approach while retaining every generated and provided baseline in the repository. It ranks models using the same public five-minute scorer; it does not choose a model from intuition or copy labels from the public ground truth. Generated result CSVs are intentionally gitignored, so a fresh checkout automatically reruns the model generators when its scoreboard is absent.

The comparison is layered: most approaches share the canonical observations and scenario detector, then differ in how they resolve missing-ball intervals, transitions, probabilities, Viterbi paths, or durations. The promoted public path is `raw tactical tracking -> temporal/missing-ball features -> preliminary scenarios -> anchors -> final scenario context -> transition/no-switch hypotheses -> interval resolver -> bounded forward-settling D evidence -> H/A/D/X labels`. George's raw observation stage is available as an upstream private-match ablation, but it is not enabled in the promoted score because evidence-only George tied the previous reference and coordinate filling reduced it. A model labelled `Kalman` is therefore a Kalman-supported possession policy, not a Kalman possession classifier. The Haiqal row is deliberately outside this shared scenario layer: it is an independent raw-observation distance/speed/dead-ball HMM baseline adapted from `origin/haiqal`. The optional Bayesian-anchor, HSMM, MLE-calibrated, EM, and supervised Kinson experiments are comparison material only; none is silently inserted into the promoted label-free path.

The two latest transition experiments are also included as diagnostics: they test whether a proposed switch should be confirmed by post-contact motion plus team-level arrival, and whether a restart should remain `D` until restart-compatible movement plus stable control. They are deliberately ranked rather than silently promoted; a lower spell score means the signal is not yet safe as a direct label rewrite. George coordinate-fill and evidence-only variants are also ranked separately so an alternate-camera observation cannot be mistaken for a possession label.

In [1]:
from pathlib import Path
import os
import subprocess
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
while not (ROOT / 'evaluation').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.environ['MTAH_DATA_DIR'] = str(ROOT / 'data')
RESULTS_DIR = ROOT / 'evaluation' / 'results'
SCOREBOARD_PATH = ROOT / 'evaluation' / 'scoreboard.csv'
PROVIDED_SCOREBOARD_PATH = ROOT / 'evaluation' / 'compare_results' / 'scoreboard.csv'
RERUN_MODELS = False  # set True to force regeneration; missing scoreboard triggers this automatically
RANK_BY = ['spell_f1_mean', 'macro_f1', 'trans_f1_25']
SHOW_ALL_GENERATED_RESULTS = False
REQUIRED_GENERATED_OUTPUT = RESULTS_DIR / 'diagnostic_forward_settling_dead_ball.csv'
if not SCOREBOARD_PATH.exists() or not REQUIRED_GENERATED_OUTPUT.exists():
    RERUN_MODELS = True
    print('Generated ranking artifacts are incomplete; 09 will run the model generators before ranking.')
GEORGE_BALLFILL = os.environ.get('GEORGE_BALLFILL')
GEORGE_EVIDENCE_ONLY = os.environ.get('GEORGE_EVIDENCE_ONLY', '1').strip().lower() in {'1', 'true', 'yes', 'on'}
GEORGE_MINIMUM_CONFIDENCE = os.environ.get('GEORGE_MINIMUM_CONFIDENCE')
print(f'RERUN_MODELS={RERUN_MODELS}')
print(f'ranking keys: {RANK_BY}')
print(f'results directory: {RESULTS_DIR}')
print('George evidence-only:', GEORGE_EVIDENCE_ONLY if GEORGE_BALLFILL else 'not applicable')
print('George minimum confidence:', GEORGE_MINIMUM_CONFIDENCE or 'none')

RERUN_MODELS=False
ranking keys: ['spell_f1_mean', 'macro_f1', 'trans_f1_25']
results directory: /Users/luke/Desktop/barca-hack-repo-staging/evaluation/results
George evidence-only: not applicable
George minimum confidence: none


## Re-run toggle

Leave `RERUN_MODELS = False` to load the committed results immediately. Set it to `True` when the feature or model code has changed; the scenario-aware experiment runners and the independent Haiqal runner will regenerate their outputs, then the common evaluator will rebuild the scoreboard. Set `GEORGE_BALLFILL` to pass George's observed/fused ball table into the canonical runner before derived features; `GEORGE_EVIDENCE_ONLY=1` is the safe default and preserves tactical coordinates while exposing source-aware evidence. The runners intentionally retain their full baseline output sets; the ranking table below only presents the final representatives.

## Contact/arrival evidence result

The current best architecture exposes a soft `contact_arrival_coherence` channel: plausible contact or motion change is checked against bounded future team arrival/reception/post-control evidence. This is deliberately an observation for the transition ledger, not a direct label rule. The earlier visible-ball reference, the local-boundary diagnostic, the probability-ledger path, and the Haiqal-combined path produced zero changed full-length labels at `0.8917 / 0.6207 / 0.4571`; the promoted forward-settling candidate then improved the public result to Macro-F1 `0.8947`, Spell-F1 `0.6400`, Transition F1@25 `0.4865`. The hard local-boundary variant remained lower at `0.8833 / 0.6069 / 0.4000`, so contact/arrival remains evidence rather than a hard rewrite. Hyperparameter search is intentionally limited to predeclared sensitivity checks; broad optimisation on this five-minute slice would overfit the 17 graded possessions.

In [2]:
RUNNER_MODULES = [
    'possession_lab.run_interpolated_flight_experiment',
    'possession_lab.run_temporal_experiment',
    'possession_lab.run_haiqal_baseline',
    'possession_lab.run_calibration_experiment',
]

if RERUN_MODELS:
    runner_environment = os.environ.copy()
    runner_environment['MTAH_DATA_DIR'] = str(ROOT / 'data')
    for module in RUNNER_MODULES:
        print(f'Running {module} ...')
        runner_args = [sys.executable, '-m', module]
        if module == 'possession_lab.run_interpolated_flight_experiment' and GEORGE_BALLFILL:
            runner_args.extend(['--george-ballfill', GEORGE_BALLFILL])
            if GEORGE_EVIDENCE_ONLY:
                runner_args.append('--george-evidence-only')
            if GEORGE_MINIMUM_CONFIDENCE:
                runner_args.extend(['--george-minimum-confidence', GEORGE_MINIMUM_CONFIDENCE])
        completed = subprocess.run(
            runner_args,
            cwd=ROOT,
            env=runner_environment,
            capture_output=True,
            text=True,
            check=False,
        )
        print(completed.stdout[-2000:])
        if completed.returncode:
            print(completed.stderr[-4000:])
            raise RuntimeError(f'{module} failed with exit status {completed.returncode}')
    evaluator = subprocess.run(
        [sys.executable, str(ROOT / 'evaluation' / 'run_evaluation.py'), '--gt', 'public', '--no-detail'],
        cwd=ROOT,
        capture_output=True,
        text=True,
        check=False,
    )
    print(evaluator.stdout[-4000:])
    if evaluator.returncode:
        print(evaluator.stderr[-4000:])
        raise RuntimeError(f'public evaluator failed with exit status {evaluator.returncode}')
else:
    print('Loading committed model outputs; no runners were started.')

Loading committed model outputs; no runners were started.


## Final representatives

The manifest is intentionally small and auditable. The four `provided baseline` rows come from the supplied comparison set. The remaining rows are one representative of the current scenario, missing-ball, interval-transition, probability-emission, Viterbi, and HSMM approaches, plus the independent Haiqal raw-observation baseline. The visible-ball NO_SWITCH row tests the new interval-level retention alternative: it keeps the prior team when the transition ledger supports no switch and no new control anchor confirms a change. Optional Bayesian-anchor and additional duration rows are included when their runner outputs exist. Haiqal does not consume scenario evidence; its row tests the value of the layered architecture against a standalone distance/speed/dead-ball HMM. All other generated CSVs remain available for audit and can be shown with `SHOW_ALL_GENERATED_RESULTS`.

In [3]:
manifest = pd.DataFrame([
    {'submission': 'stage_hq_viterbi', 'approach': 'provided baseline', 'decision_layer': 'provided Viterbi decoder'},
    {'submission': 'evidence_bounded_deadball', 'approach': 'provided baseline', 'decision_layer': 'provided evidence-bounded deadball'},
    {'submission': 'aligned_physics_veto', 'approach': 'provided baseline', 'decision_layer': 'provided aligned physics veto'},
    {'submission': 'protected_aligned_reception', 'approach': 'provided baseline', 'decision_layer': 'provided protected reception'},
    {'submission': 'diagnostic_visible_ball_low_motion_dead_ball', 'approach': 'scenario + flight', 'decision_layer': 'visible-ball low-motion deadball candidate'},
    {'submission': 'diagnostic_visible_ball_no_switch_interval', 'approach': 'scenario + interval transition', 'decision_layer': 'visible-ball deadball + NO_SWITCH interval resolver'},
    {'submission': 'baseline_scenario_bidirectional_interpolated_flight_ensemble', 'approach': 'scenario + flight', 'decision_layer': 'bidirectional scenario/interpolated flight ensemble'},
    {'submission': 'baseline_same_team_missing_ball_trajectory', 'approach': 'missing-ball geometry', 'decision_layer': 'same-team trajectory bridge'},
    {'submission': 'baseline_same_team_kalman_missing_ball_trajectory', 'approach': 'missing-ball geometry', 'decision_layer': 'same-team Kalman-supported trajectory bridge'},
    {'submission': 'baseline_transition_interval_hypothesis', 'approach': 'interval transition', 'decision_layer': 'explicit transition interval hypothesis'},
    {'submission': 'baseline_trusted_transition', 'approach': 'interval transition', 'decision_layer': 'trusted transition resolver'},
    {'submission': 'baseline_transition_ledger_viterbi', 'approach': 'Viterbi', 'decision_layer': 'transition-ledger Viterbi'},
    {'submission': 'baseline_probability_transition_ledger_viterbi', 'approach': 'probability + Viterbi', 'decision_layer': 'normalized evidence-emission Viterbi'},
    {'submission': 'baseline_scenario_confidence_duration', 'approach': 'HSMM/duration', 'decision_layer': 'scenario confidence duration decoder'},
    {'submission': 'baseline_restart_type_specific_interpolated_flight', 'approach': 'restart + flight', 'decision_layer': 'restart-type-specific interpolated flight'},
    {'submission': 'baseline_haiqal_raw_viterbi', 'approach': 'independent raw HMM', 'decision_layer': 'Haiqal distance/speed/dead-ball clues + Viterbi'},
    {'submission': 'diagnostic_no_switch_true_switch_confirmation', 'approach': 'transition signal diagnostic', 'decision_layer': 'NO_SWITCH + post-contact/team-level switch confirmation'},
    {'submission': 'diagnostic_no_switch_restart_activation', 'approach': 'transition signal diagnostic', 'decision_layer': 'NO_SWITCH + restart-compatible control activation'},
    {'submission': 'diagnostic_no_switch_true_switch_restart_activation', 'approach': 'transition signal diagnostic', 'decision_layer': 'NO_SWITCH + switch confirmation + restart activation'},
    {'submission': 'diagnostic_no_switch_hysteretic_handoff', 'approach': 'transition signal diagnostic', 'decision_layer': 'NO_SWITCH + delayed stable-anchor handoff'},
    {'submission': 'diagnostic_no_switch_restart_state_machine', 'approach': 'transition signal diagnostic', 'decision_layer': 'NO_SWITCH + bounded restart state machine'},
    {'submission': 'diagnostic_no_switch_restart_hysteretic_handoff', 'approach': 'transition signal diagnostic', 'decision_layer': 'restart state machine + delayed stable-anchor handoff'},
    {'submission': 'diagnostic_no_switch_restart_phase_interval', 'approach': 'restart phase + interval transition', 'decision_layer': 'restart evidence -> transition constraints -> shared interval resolver'},
    {'submission': 'diagnostic_no_switch_restart_phase_boundary_guard05', 'approach': 'local restart boundary diagnostic', 'decision_layer': '5-frame restart boundary guard'},
    {'submission': 'diagnostic_no_switch_restart_phase_boundary_guard12', 'approach': 'local restart boundary diagnostic', 'decision_layer': '12-frame restart boundary guard'},
    {'submission': 'diagnostic_no_switch_restart_phase_boundary_guard25', 'approach': 'local restart boundary diagnostic', 'decision_layer': '25-frame restart boundary guard'},
])
optional_manifest = pd.DataFrame([
    {'submission': 'baseline_haiqal_evidence_probability_ledger_viterbi', 'approach': 'our pipeline + Haiqal evidence', 'decision_layer': 'Haiqal raw clues + our probability/transition ledger + Viterbi'},
    {'submission': 'diagnostic_haiqal_visible_ball_no_switch_interval', 'approach': 'our best + Haiqal evidence', 'decision_layer': 'Haiqal closeness observation + visible-ball D + NO_SWITCH interval resolver'},
    {'submission': 'diagnostic_visible_ball_no_switch_bootstrap_guard', 'approach': 'nonparametric stability guard', 'decision_layer': 'frame bootstrap lower bound + NO_SWITCH interval resolver'},
    {'submission': 'diagnostic_george_fused_observed_no_switch_interval', 'approach': 'our best + George observations (ablation)', 'decision_layer': 'George fused observed ball gaps + visible-ball D + NO_SWITCH interval resolver'},
    {'submission': 'diagnostic_local_touch_viterbi', 'approach': 'local short-touch diagnostic', 'decision_layer': 'bounded local Viterbi over control/short-touch paths'},
    {'submission': 'diagnostic_local_short_touch_attribution', 'approach': 'local short-touch diagnostic', 'decision_layer': 'bounded proximity/contact/backtrack/forward episode resolver'},
    {'submission': 'diagnostic_local_short_touch_attribution_strict', 'approach': 'local short-touch sensitivity', 'decision_layer': 'strict bounded proximity/contact/forward episode resolver'},
    {'submission': 'diagnostic_restart_long_ball_origin_landing', 'approach': 'restart + long-ball continuity diagnostic', 'decision_layer': 'launching-team continuity during flight + persistent landing control'},
    {'submission': 'diagnostic_channel_profile_anchor_dominant', 'approach': 'channel composition diagnostic', 'decision_layer': 'anchor-dominant H/A support + NO_SWITCH interval resolver'},
    {'submission': 'diagnostic_channel_profile_balanced', 'approach': 'channel composition diagnostic', 'decision_layer': 'balanced H/A support + NO_SWITCH interval resolver'},
    {'submission': 'diagnostic_channel_profile_independent_support', 'approach': 'channel composition diagnostic', 'decision_layer': 'independent motion/arrival support + NO_SWITCH interval resolver'},
    {'submission': 'diagnostic_resolver_profile_conservative_long', 'approach': 'resolver sensitivity', 'decision_layer': 'conservative-long NO_SWITCH interval resolver'},
    {'submission': 'diagnostic_resolver_profile_conservative', 'approach': 'resolver sensitivity', 'decision_layer': 'conservative NO_SWITCH interval resolver'},
    {'submission': 'diagnostic_resolver_profile_balanced_current', 'approach': 'resolver sensitivity', 'decision_layer': 'balanced-current NO_SWITCH interval resolver'},
    {'submission': 'diagnostic_resolver_profile_aggressive', 'approach': 'resolver sensitivity', 'decision_layer': 'aggressive NO_SWITCH interval resolver'},
    {'submission': 'diagnostic_resolver_profile_aggressive_short', 'approach': 'resolver sensitivity', 'decision_layer': 'aggressive-short NO_SWITCH interval resolver'},
    {'submission': 'baseline_bayesian_hmm', 'approach': 'Bayesian HMM', 'decision_layer': 'Haiqal-style Dirichlet anchor transition prior + Viterbi'},
    {'submission': 'baseline_hmm_anchor', 'approach': 'HMM', 'decision_layer': 'anchor-initialized diagonal-Gaussian HMM'},
    {'submission': 'baseline_hsmm_anchor', 'approach': 'HSMM/duration', 'decision_layer': 'anchor-initialized duration cleanup'},
    {'submission': 'baseline_scenario_hsmm', 'approach': 'HSMM/duration', 'decision_layer': 'scenario-aware semi-Markov decoder'},
])

current_scores = pd.read_csv(SCOREBOARD_PATH).assign(score_source='current runners')
provided_scores = pd.read_csv(PROVIDED_SCOREBOARD_PATH).assign(score_source='provided comparison set')
scores = pd.concat([current_scores, provided_scores], ignore_index=True, sort=False)
scores = scores.drop_duplicates(subset=['submission'], keep='first')
manifest = pd.concat([manifest, optional_manifest[optional_manifest['submission'].isin(scores['submission'])]], ignore_index=True)
ranked = manifest.merge(scores, on='submission', how='left', validate='one_to_one')
missing = ranked.loc[ranked['macro_f1'].isna(), 'submission'].tolist()
if missing:
    raise FileNotFoundError(f'Manifest outputs are not scored: {missing}')
ranked = ranked.sort_values(RANK_BY, ascending=False).reset_index(drop=True)
ranked = ranked.drop(columns=['rank'], errors='ignore')
ranked.insert(0, 'rank', ranked.index + 1)
ranking_columns = ['rank', 'approach', 'decision_layer', 'submission', 'score_source', 'macro_f1', 'spell_f1_mean', 'trans_f1_25', 'f1_H', 'f1_A', 'f1_D']
display(ranked[ranking_columns])

if SHOW_ALL_GENERATED_RESULTS:
    all_columns = [c for c in ['rank', 'submission', 'macro_f1', 'spell_f1_mean', 'trans_f1_25', 'f1_H', 'f1_A', 'f1_D'] if c in current_scores]
    display(current_scores.sort_values(['spell_f1_mean', 'macro_f1'], ascending=False)[all_columns])

,rank,approach,decision_layer,submission,score_source,macro_f1,spell_f1_mean,trans_f1_25,f1_H,f1_A,f1_D
0,1,scenario + interval transition,visible-ball deadball + NO_SWITCH interval res...,diagnostic_visible_ball_no_switch_interval,current runners,0.891747,0.620690,0.457143,0.913319,0.852768,0.909156
1,2,transition signal diagnostic,NO_SWITCH + delayed stable-anchor handoff,diagnostic_no_switch_hysteretic_handoff,current runners,0.891747,0.620690,0.457143,0.913319,0.852768,0.909156
2,3,restart phase + interval transition,restart evidence -> transition constraints -> ...,diagnostic_no_switch_restart_phase_interval,current runners,0.891747,0.620690,0.457143,0.913319,0.852768,0.909156
3,4,our best + Haiqal evidence,Haiqal closeness observation + visible-ball D ...,diagnostic_haiqal_visible_ball_no_switch_interval,current runners,0.891747,0.620690,0.457143,0.913319,0.852768,0.909156
4,5,local short-touch diagnostic,bounded local Viterbi over control/short-touch...,diagnostic_local_touch_viterbi,current runners,0.891747,0.620690,0.457143,0.913319,0.852768,0.909156
5,6,channel composition diagnostic,anchor-dominant H/A support + NO_SWITCH interv...,diagnostic_channel_profile_anchor_dominant,current runners,0.891747,0.620690,0.457143,0.913319,0.852768,0.909156
6,7,channel composition diagnostic,balanced H/A support + NO_SWITCH interval reso...,diagnostic_channel_profile_balanced,current runners,0.891747,0.620690,0.457143,0.913319,0.852768,0.909156
7,8,channel composition diagnostic,independent motion/arrival support + NO_SWITCH...,diagnostic_channel_profile_independent_support,current runners,0.891747,0.620690,0.457143,0.913319,0.852768,0.909156
8,9,resolver sensitivity,conservative-long NO_SWITCH interval resolver,diagnostic_resolver_profile_conservative_long,current runners,0.891747,0.620690,0.457143,0.913319,0.852768,0.909156
9,10,resolver sensitivity,conservative NO_SWITCH interval resolver,diagnostic_resolver_profile_conservative,current runners,0.891747,0.620690,0.457143,0.913319,0.852768,0.909156


## Haiqal evidence ablation

This compares the independent Haiqal output with two integration boundaries. The probability-ledger variant imports only Haiqal's raw distance, speed, visibility, out-of-pitch, and dead-ball clues; the best-model variant sends only the bounded closeness observation into our transition-trust layer. Neither imports Haiqal's final labels or replaces our resolver. The best-model combination changed 0 full-match frames and tied the earlier visible-ball reference exactly (`0.891747` Macro F1 / `0.620690` Spell F1 / `0.457143` Transition F1@25), so it did not improve the promoted forward-settling candidate. That is a useful negative result: Haiqal's distance signal is already represented by our nearest-player/reception evidence, or is too weak to alter a guarded decision. It should remain an available observation adapter, but it is not promoted as an ensemble gain. The latest `origin/haiqal` commit (`5092f12`) was also run directly: its raw model produces the same public metrics as our adapter (`0.7677` / `0.5143` / `0.2667`), with only five implementation-level boundary frames differing from the adapter because of numeric/loading details.

## Transition-signal ablation

The runner also evaluates two promising, label-free signals from the disagreement audit. `true_switch_confirmation` asks for post-contact ball deceleration/direction change, team-level receiver arrival, and persistence of the new controlled state. `restart_activation` keeps `D` until restart-compatible ball movement agrees with stable team-level control. Both are composed from observations already available to the pipeline; neither reads public labels during inference. On this five-minute evaluation slice, the direct rewrites were not promoted:

| Diagnostic | Macro F1 | Spell F1 | Transition F1@25 | Interpretation |
|---|---:|---:|---:|---|
| NO_SWITCH + true-switch confirmation | 0.8866 | 0.3184 | 0.3273 | switch confirmation was too aggressive and fragmented spells |
| NO_SWITCH + restart activation | 0.8234 | 0.5600 | 0.3429 | restart boundary timing improved over some baselines but lost live-state continuity |
| NO_SWITCH + both signals | 0.8183 | 0.2880 | 0.2545 | composing direct rewrites amplified both failure modes |

The result is useful rather than a failure: these observations remain candidates for a guarded evidence layer, but the functions should not directly overwrite labels. The current best remains `diagnostic_visible_ball_no_switch_interval`; the next experiment should use the signals to raise or lower transition trust inside an interval resolver and preserve an explicit no-switch/contested alternative.

### Guarded handoff and restart-state diagnostics

A delayed handoff that requires a later stable anchor produced exactly the same output as the current best (`0.8917` / `0.6207` / `0.4571`), so it is a safe no-op rather than a gain. The bounded restart state machine improved the direct restart rewrite to `0.8245` / `0.5931` / `0.3529`, but it remains below the active model; adding delayed handoff did not change that result. These diagnostics therefore stay in the appendix.

The new `diagnostic_no_switch_restart_phase_interval` candidate changes the architecture rather than the score: it emits `PRE_RESTART`, `ACTIVE_UNCONTROLLED`, and `CONTROL_CONFIRMED` evidence, blocks an unconfirmed H/A handoff, and releases `D` only after sustained independent team control. It ties the current best exactly on the public slice (`0.8917` / `0.6207` / `0.4571`) and changes only frames outside the 7,500 graded public frames. It is therefore a structural full-match diagnostic, not a claimed metric improvement.

The local-boundary guards test the narrower alternative: constrain only the first proposed restart handoff instead of suppressing the entire phase. The 5-frame guard is the least disruptive (628 full-match frames changed) but still does not improve the active public score; 12 and 25 frames are progressively worse on transition timing. These remain diagnostic, and the original NO_SWITCH resolver remains the private baseline.

### Nonparametric transition stability

The runner now writes `transition_bootstrap_stability.csv`. For each inferred interval it resamples the underlying per-frame trust, activity, contest, dead-ball, and uncertainty evidence, reconstructs the interval scores, and records how often `H`, `A`, `NO_SWITCH`, or `CONTESTED` wins. This is an empirical stability estimate, not a calibrated probability of the hidden label. The channel report `transition_channel_stability.csv` separately leaves out anchors, ball motion/contact, player arrival, missing-ball cause, restart context, and Haiqal observations.

On this run, 13 of 1,306 intervals had bootstrap state stability below 0.80. Applying the 2.5th-percentile no-switch bound as a hard label guard changed 1,671 full-match frames and reduced the public result to Macro F1 `0.888744`, Spell F1 `0.600000`, and Transition F1@25 `0.444444`, below the current best `0.891747` / `0.620690` / `0.457143`. Therefore bootstrap is retained as an ambiguity and audit signal, not promoted as a label rewrite. The channel audit is the better next use: it identifies which evidence families are indispensable without using ground truth.

In [4]:
ablation_names = [
    'baseline_haiqal_raw_viterbi',
    'baseline_probability_transition_ledger_viterbi',
    'baseline_haiqal_evidence_probability_ledger_viterbi',
    'diagnostic_visible_ball_low_motion_dead_ball',
]
ablation_columns = [c for c in ['submission', 'macro_f1', 'spell_f1_mean', 'trans_f1_25'] if c in current_scores]
display(current_scores[current_scores['submission'].isin(ablation_names)][ablation_columns].sort_values('submission'))

,submission,macro_f1,spell_f1_mean,trans_f1_25
54,baseline_haiqal_evidence_probability_ledger_vi...,0.771958,0.567742,0.333333
111,baseline_haiqal_raw_viterbi,0.767720,0.514286,0.266667
57,baseline_probability_transition_ledger_viterbi,0.771958,0.567742,0.333333
36,diagnostic_visible_ball_low_motion_dead_ball,0.888744,0.600000,0.444444


## Calibration diagnostics (reported separately from the model ranking)

MLE temperature scaling is not a possession submission and is not ranked by macro-F1/spell-F1. It is evaluated with log loss and Brier score on a disjoint public holdout. A worse holdout score is useful evidence of overfitting or distribution shift, not a reason to hide the experiment.

In [5]:
calibration_path = ROOT / 'evaluation' / 'reports' / 'probability_calibration_public_holdout.csv'
if calibration_path.exists():
    display(pd.read_csv(calibration_path))
else:
    print('Calibration report is absent; set RERUN_MODELS=True to generate it.')

bootstrap_path = ROOT / 'evaluation' / 'reports' / 'transition_bootstrap_stability.csv'
if bootstrap_path.exists():
    bootstrap = pd.read_csv(bootstrap_path)
    display(bootstrap[['stable_state', 'stable_state_fraction', 'no_switch_evidence_lower']].describe())

channel_path = ROOT / 'evaluation' / 'reports' / 'transition_channel_stability.csv'
if channel_path.exists():
    channel = pd.read_csv(channel_path)
    display(channel.groupby('channel').agg(intervals=('interval_id', 'size'), agreement=('state_agrees', 'mean'), changed_intervals=('state_agrees', lambda values: int((~values).sum()))).reset_index())

,split,model,temperature,log_loss,brier
0,holdout,rule_evidence_softmax,1.000000,1.366923,0.699768
1,holdout,mle_temperature_scaled,0.778684,1.523411,0.741228


,stable_state_fraction,no_switch_evidence_lower
count,1306.000000,1306.000000
mean,0.996486,0.624934
std,0.029975,0.134636
min,0.558594,0.235596
25%,1.000000,0.575758
50%,1.000000,0.638368
75%,1.000000,0.728520
max,1.000000,0.877401


,channel,intervals,agreement,changed_intervals
0,anchors,1306,0.976263,31
1,ball_motion_contact,1306,0.816998,239
2,haiqal,1306,0.980092,26
3,missing_ball_cause,1306,0.980092,26
4,player_arrival,1306,0.899694,131
5,restart_context,1306,0.980092,26


## How to interpret the approaches

- Scenario detection is shared evidence for most current models; it is not something to exclude when testing transitions.
- The probability model is a normalized evidence-emission variant, not calibrated probability and not a replacement for scenario detection.
- MLE temperature scaling is supervised calibration only when fitted on a declared reviewed-label split; it must be scored on a disjoint holdout.
- Haiqal's Bayesian-anchor model uses a Dirichlet-smoothed transition prior from algorithmic anchors, not public labels; it is a useful transition-prior comparison, not calibrated softmax output.
- EM is an unsupervised latent-state experiment and must be checked for label switching, anchor violations, and stability before promotion.
- Viterbi is a path smoother over those emissions and transition hypotheses; it is not the primary truth-maker.
- HSMM adds duration assumptions; it remains a comparison approach until duration distributions are learned from reviewed labels.
- Kalman is tested as missing-ball geometry support. Its score answers whether the bridge helps the complete policy, not whether Kalman alone labels possession.
- Rankings on the five-minute public slice are evidence, not universal truth; spell and transition scores are especially noisy on this short benchmark.
- The latest signal ablation demonstrates why ranking matters: plausible observations can be valuable for explaining a transition while still being unsafe as a hard relabelling rule. Keep the evidence, reject the direct rewrite, and test it again inside the resolver.

## Explicit transition-channel composition (diagnostic)

The runner now tests the proposed architecture directly: George observed-only ball evidence enters before our derived features; Haiqal contributes bounded auxiliary observations; then an explicit channel profile combines anchor agreement, ball motion/contact, player arrival, restart continuity, and bounded Haiqal support before the interval resolver chooses among H/A/NO_SWITCH/CONTESTED. Neither collaborator's final labels are imported, and no public labels are used to fit the weights.

The three predeclared profiles—`anchor_dominant`, `balanced`, and `independent_support`—all produced zero label changes relative to the previous visible-ball + `NO_SWITCH` candidate, including zero changes on public scored frames. Each scored Macro-F1 `0.891747`, Spell-F1 `0.620690`, and Transition F1@25 `0.457143`; they therefore do not replace the promoted forward terminal-flight candidate. They demonstrate that the guarded resolver is stable under this bounded evidence reweighting on the five-minute slice.

The channel dropout audit is more informative than the tied score: removing ball-motion/contact changed 283 inferred intervals, player-arrival evidence 147, anchors 37, Haiqal 33, and restart context 33. These are sensitivity measurements, not correctness measurements. The next signal-led improvement should focus on improving contact and team-arrival observations on the private full match, not increasing Haiqal's weight or forcing restart labels.

## Contact/arrival signal experiment

The leave-one-out audit identified ball-motion/contact and player-arrival evidence as the strongest transition channels. We tested a structural improvement rather than increasing their weights: team-level arrival consensus and two-sided missing-gap trajectory continuity.

| Candidate | Macro-F1 | Spell-F1 | Transition F1@25 | Decision |
|---|---:|---:|---:|---|
| Promoted: forward terminal-flight + `NO_SWITCH` interval | 0.8947 | 0.6400 | 0.4865 | winner |
| Local contact/arrival handoff guard | 0.8833 | 0.6207 | lower than current | appendix |

The rejected guard changed 1,609 full-match H/A frames (779 H→A and 830 A→H). That is not a harmless boundary adjustment; it shows that genuine handoffs are sometimes weakly observed and confirmed later. The architecture therefore remains `observations → scenarios/anchors → transition hypotheses → interval resolver → H/A/D/X`, with consensus and continuity as soft evidence. Public labels were used only after the run for evaluation, never to construct the signals.

## Kinson supervised experiment (appendix-only)

**Problem.** Kinson's standalone supervised H/A/D + Viterbi path is a different decoder and can fragment spells when its training distribution or silver labels do not match this tactical camera. It should not replace our anchors or interval resolver.

**Decision.** Do not add Kinson's fitted probabilities or Viterbi path to the promoted resolver. They may be ranked as a separate supervised comparison when the training matches and label provenance are declared, but they are not part of the label-free organiser-facing method.

**Construction.** Kinson's trainer builds features from multiple matches, holds out one match at a time, tunes `switch_cost`, `dead_cost`, and minimum spell durations on out-of-fold labelled predictions, then fits a final classifier. The labels are event-derived silver labels rather than the hackathon ground truth, but that still makes this supervised/weakly supervised learning. It remains a separate comparison because those fitted probabilities are not part of our label-free resolver.

**Public result.** The standalone local Kinson path scored Macro-F1 `0.6425`, Spell-F1 `0.3590`, and Transition F1@25 `0.2667`. The bounded integration changed no labels and tied the current best, so it supplied no measured gain. It is retained as an appendix comparison and excluded from the promoted pipeline because its fitted probabilities depend on labelled training data.

## Final resolver and robustness conclusion

**Problem → idea → construction → evaluation.** The public disagreement audit contains ambiguous live-ball and restart boundaries, so the remaining decision is not which threshold matches the public labels but how much continuity protection to use on unseen frames. We therefore ran five predeclared resolver profiles from conservative-long to aggressive-short. They changed only the interval no-switch gate, support fraction, maximum interval length, and prior-run requirement.

All five profiles scored Macro-F1 **0.8917**, Spell-F1 **0.6207**, and Transition F1@25 **0.4571** on the public slice. The conservative profiles produced the exact previous public labels; the aggressive profiles changed 561–674 full-timeline frames, but none was in the public scored window. The public set cannot identify the private-optimal aggressiveness, so the **balanced-current** resolver remains the transition policy beneath the promoted terminal-flight D-onset layer.

Density, pressure, closing speed, pass options, significant-player counts, and landing-area crowding were audited as transition-risk signals. They are higher in disagreement frames, especially in contested/unknown/flight contexts. We use them to lower transition trust or preserve CONTESTED; they do not directly assign H or A because the same pattern can mean pressured retention, deflection, or a genuine turnover.

The final candidate is therefore: raw tracking → our temporal/missing-ball features → preliminary observation scenarios → anchors → final scenario context → H/A/NO_SWITCH/CONTESTED hypotheses → shared interval resolver → bounded forward terminal-flight/scene-settling D onset → H/A/D/X labels. George's raw observation stage can be inserted before temporal features for a separate private-match ablation, but it is not part of the promoted public winner. The promoted candidate is `diagnostic_forward_settling_dead_ball`, scoring Macro-F1 `0.894673`, Spell-F1 `0.640000`, and Transition F1@25 `0.486486`. The available local George coordinate-fill CSV changed 7,751 final labels and reduced the public result to `0.8723 / 0.6000 / 0.3889`, so coordinate replacement is not enabled in the winner; evidence-only George remains available for auditing. The new IDSSE-backed George HMM-only experiment uses Kloppy-parsed open matches to create carrier/dead-ball posteriors, but passes them only as bounded hidden-path evidence. It changed 373 full-length labels, none on missing-ball public frames, and scored `0.8924 / 0.6400 / 0.4324`; it is retained as a robustness artifact, not selected by the five-minute score. Haiqal and Kinson are not promoted inputs.

## Local short-touch diagnostic: result

**Problem → idea → construction → evaluation.** Two disagreement families require opposite corrections: frames `11351:11497` contain a delayed true handoff, while frames `11553:11623` and `15656:15761` contain early false H/A handoffs. We therefore tested a bounded local Viterbi path over control and short-touch states, using contact, arrival, flight, contested, and later-control evidence.

The diagnostic changed **0 full-length frames**, including the four reviewed windows, and tied the previous visible-ball reference at Macro-F1 **0.8917**, Spell-F1 **0.6207**, and Transition F1@25 **0.4571**. It remains below the promoted forward terminal-flight candidate. This is not evidence that Viterbi is useless; it shows that the current observation channels do not yet provide a reliable distinction between a short controlled touch and a deflection/nearest-player artefact. The candidate remains an appendix diagnostic. The next useful signal is better contact/arrival/control evidence, not more decoder tuning.

## Promoted signal: forward terminal-flight dead-ball onset

**Problem → idea → construction → evaluation.** The previous resolver detected some `D` intervals only after a trailing low-motion run. In the reviewed disagreement around frames `15889:15993`, the ball was visible and slowing, but its path was almost straight; acceleration and direction change alone did not provide a reliable boundary.

The new bounded evidence layer starts from the resolved `NO_SWITCH` output and looks forward up to 75 frames. It requires a visible terminal-flight speed drop, future player activity settling, and separation from the nearest plausible receiver. Rolling and forward direction-change statistics are retained as motion evidence, but a direction change alone never creates `D`. This is an observation-first rule, not a label fitted to the public ground truth.

The balanced candidate changed 82 full-length frames. In the graded public slice, 19 of 22 changed frames corrected the late `A → D` onset and 3 were early by three frames. It improved the previous winner from `0.891747 / 0.620690 / 0.457143` to Macro-F1 **0.894673**, Spell-F1 **0.640000**, and Transition F1@25 **0.486486**. A 90-frame and a stricter predeclared profile tied the previous score, so they were not promoted. The ungraded full-match changes remain subject to private-match review.

## Transition roles: contact, hypothesis, control, and restart

**Problem.** A single transition flag was carrying several different football questions: did the ball experience contact, did a touch possibly change team, has a team actually controlled the ball, and has a restart become live? This matters when a dead ball is followed by a long ball: the launching team can be identified, but the landing team is not known until later control evidence.

**Idea.** The feature layer now exposes separate, label-free roles: `CONTACT_ANCHOR`, `SWITCH_HYPOTHESIS`, `CONTROL_ANCHOR`, `RESTART_ACTIVATION`, `RESTART_PENDING`, `RESTART_LONG_BALL`, and bounded `D_HOLD`. A contact or restart launch does not directly overwrite H/A/D. A long-ball origin is a continuity anchor for the initiating team, not a claim that it controls the landing.

**Construction.** Contact combines deceleration, robust direction change, kick/contact motion, and reception signals. A switch hypothesis comes from interval evidence and opposing-anchor direction. Control requires a high-confidence team anchor together with controlled-reception or post-control evidence. Restart activation requires restart context plus physical ball movement; `D_HOLD` is limited to stoppage/low-activity evidence without activation or control. All fields are generated before public labels are joined.

**Evaluation.** The public run writes `evaluation/reports/transition_role_evidence.csv`, `transition_role_summary.csv`, and role fields in `transition_hypothesis_ledger.csv`. This pass improved observability but did not change the promoted submission, so the measured score remains Macro-F1 **0.8947**, Spell-F1 **0.6400**, and Transition F1@25 **0.4865**. That is intentional: a role is promoted into the label resolver only after a predeclared, label-free candidate improves the public audit without making private-match assumptions more brittle.

## Missing-ball path hypotheses: upstream evidence, not a second decoder

**Problem → idea → construction.** A missing ball can hide a straight flight, a player-supported deflection, or a stoppage/resting interval. Treating every gap as interpolation, or treating every direction change as a possession switch, would manufacture transitions. The feature layer therefore runs this test only on two-sided missing-ball gaps.

It compares three label-free scores: `STRAIGHT`, `PLAYER_DEFLECTION`, and `DEAD_RESTING`. The scores use gap-edge direction change, Kalman/George path agreement and uncertainty, nearby-player arrival, pressure, density, and post-gap control. A direction change alone is never enough. The output is a path hypothesis plus confidence/uncertainty that scenarios and transition hypotheses may inspect; the interval resolver still owns `H`, `A`, `NO_SWITCH`, `CONTESTED`, and final `H/A/D/X`.

This is distinct from George's carrier/dead-ball HMM: George supplies optional path continuity and provenance, while this layer asks whether the observed evidence supports a deflection alternative. Kalman remains a short-gap trajectory estimator, not a possession classifier. The public runner is used to test whether the extra evidence changes the promoted result; it is not promoted merely because it explains a disagreement.

## Appendix — local short-touch attribution diagnostic

**Problem.** A brief opponent touch can occur inside a longer possession spell, so a global threshold or an automatic `CONTESTED → H/A` rewrite would overreact. The public disagreement audit motivated a local test for opposing-player proximity, ball contact/direction change, backward approach evidence, and a few frames of forward persistence.

**Construction.** The experiment adds these observations without reading ground truth. A bounded resolver may rewrite only a short live-play episode, never `D`/`X` or a high-confidence anchor, and only when the base path returns to the prior team afterwards. A stricter sensitivity version requires stronger contact, proximity, and forward persistence.

**Evaluation.** On the public five-minute slice, applying the rule to the promoted model scored Macro-F1 `0.8909`, Spell-F1 `0.3429`, and Transition F1@25 `0.4444`; the strict version scored `0.8878`, `0.2171`, and `0.4156`. The promoted reference remains `0.8947 / 0.6400 / 0.4865`. Neither variant recovered the reviewed short Home episode at frames `15766–15798`; both introduced many short spells.

**Decision.** Do not promote this rule. The result is evidence that proximity plus contact change is not yet sufficient to attribute a short touch. The missing signal is reliable player-ball contact/arrival identity, not more threshold tuning. The code remains as an auditable appendix diagnostic in `possession_lab/decoder_experiments.py` and the runner.

## Planned 02 — direction-gated George/Kalman path hypotheses

The implemented experiment uses the full-match alternate-camera assets only upstream of our feature layer. Kloppy is used to parse the locally available IDSSE training files with dead frames retained (`only_alive=False`); no hackathon public labels are used to fit the George HMM. If edge observations support a straight path, the tactical path remains primary and George contributes continuity/uncertainty. If a direction change is plausible, the existing path layer compares straight flight, player-supported deflection, and dead/resting hypotheses against player arrival, pressure, and post-gap control. Kalman remains a short, two-sided, low-uncertainty trajectory estimate; it cannot identify a hidden deflection without independent player-motion/contact evidence. The HMM posterior is passed as path evidence, never as a possession label. The public comparison showed no gain and no changed missing-ball scored frames, so this is a reproducible private-robustness option rather than a promoted rule.